In [ ]:
# %% 0) Green/yellow apple project setup — automatic pseudo-label workflow
# Video -> exactly 800 frames -> automatic YOLO apple labels -> temporal split -> safe augmentation
# -> AAM + EBCA + FM + KWConv -> AdaBins relative depth -> ByteTrack-style tracking/counting.
from pathlib import Path
import csv
import json
import math
import sys

cwd = Path.cwd().resolve()
ROOT = next(
    candidate
    for candidate in (cwd, *cwd.parents)
    if (candidate / "Dataset").exists() and (candidate / "Codeing" / "fruit_pipeline.py").exists()
)
CODE_DIR = ROOT / "Codeing"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import cv2
import numpy as np
from tqdm.auto import tqdm
import fruit_pipeline as fp

paths = fp.project_paths(ROOT)
CLASS_NAMES = fp.CLASS_NAMES  # 0=green_apple, 1=yellow_apple
print("[root]", ROOT)
print("[video]", paths.raw_video)
print("[classes]", CLASS_NAMES)


In [ ]:
# %% 1) Video preflight
preflight = fp.video_preflight(paths.raw_video)
assert preflight["playable"], preflight
print(json.dumps(preflight, indent=2))


In [ ]:
# %% 2) Extract exactly 800 temporally uniform frames from DJI_0418.MP4
# 13 samples/sec is enough to cover the ~64-second video, then the extractor selects exactly 800 uniformly.
# Set REBUILD_FRAME_SET=True once to replace stale generated raw frames (it never touches the MP4 source).
TARGET_FRAMES = 800
REBUILD_FRAME_SET = False

current_raw_count = len(list(paths.frames.glob("*.jpg")))
if current_raw_count not in (0, TARGET_FRAMES) and not REBUILD_FRAME_SET:
    raise RuntimeError(
        f"Found {current_raw_count} raw frames, not {TARGET_FRAMES}. "
        "Set REBUILD_FRAME_SET=True to recreate the generated 800-frame set."
    )

frame_manifest = fp.extract_uniform_frames(
    paths,
    sample_fps=13.0,
    max_frames=TARGET_FRAMES,
    overwrite=REBUILD_FRAME_SET,
)
raw_frames = sorted(paths.frames.glob("*.jpg"))
assert len(frame_manifest) == TARGET_FRAMES and len(raw_frames) == TARGET_FRAMES
print(f"[extract] exactly {len(raw_frames)} frames -> {paths.frames}")


In [ ]:
# %% 3) Automatic apple labelling (same pseudo-labelling idea as the reference notebook)
# A pretrained YOLO detector proposes only the COCO 'apple' class. Each apple crop is then classified
# as green or yellow using its central HSV colour evidence. No manual labelling is used.
# Generated labels: Dataset/labels/auto/*.txt | diagnostic overlays: Dataset/labels/auto_previews/.
from ultralytics import YOLO

AUTO_CONFIDENCE = 0.20
AUTO_IMAGE_SIZE = 1024
AUTO_DEVICE = 0 if __import__("torch").cuda.is_available() else "cpu"
REBUILD_AUTO_LABELS = False

# Use an already trained two-colour model on later runs; the first run uses the installed COCO detector.
trained_candidate = paths.training_results / "yolov8_green_yellow_apple" / "weights" / "best.pt"
AUTO_MODEL = trained_candidate if trained_candidate.exists() else ROOT / "yolov8n.pt"
if not AUTO_MODEL.exists() and AUTO_MODEL.name == "yolov8n.pt":
    AUTO_MODEL = "yolov8n.pt"  # Ultralytics downloads the standard COCO checkpoint if absent.

def _normalise_model_names(names):
    return {int(index): str(name).lower().strip().replace(" ", "_") for index, name in names.items()}

def classify_box_colour(image_bgr, xyxy):
    # Return 0=green or 1=yellow from the central fruit area, plus reproducibility scores.
    height, width = image_bgr.shape[:2]
    x1, y1, x2, y2 = (int(round(value)) for value in xyxy)
    margin_x, margin_y = max(1, int((x2 - x1) * 0.18)), max(1, int((y2 - y1) * 0.18))
    x1, x2 = max(0, x1 + margin_x), min(width, x2 - margin_x)
    y1, y2 = max(0, y1 + margin_y), min(height, y2 - margin_y)
    crop = image_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return fp.GREEN_APPLE, {"green_fraction": 0.0, "yellow_fraction": 0.0}
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    coloured = (hsv[..., 1] >= 35) & (hsv[..., 2] >= 40)
    denominator = max(1, int(coloured.sum()))
    green = coloured & (hsv[..., 0] >= 36) & (hsv[..., 0] <= 90)
    yellow = coloured & (hsv[..., 0] >= 14) & (hsv[..., 0] < 36)
    green_fraction = float(green.sum() / denominator)
    yellow_fraction = float(yellow.sum() / denominator)
    class_id = fp.YELLOW_APPLE if yellow_fraction > green_fraction else fp.GREEN_APPLE
    return class_id, {"green_fraction": round(green_fraction, 5), "yellow_fraction": round(yellow_fraction, 5)}

def automatic_apple_labels(model_source, *, overwrite=False):
    paths.auto_labels.mkdir(parents=True, exist_ok=True)
    paths.auto_previews.mkdir(parents=True, exist_ok=True)
    if overwrite:
        for folder, pattern in ((paths.auto_labels, "*.txt"), (paths.auto_previews, "*.jpg")):
            for generated_file in folder.glob(pattern):
                generated_file.unlink()
    model = YOLO(str(model_source))
    model_names = _normalise_model_names(model.names)
    apple_ids = {index for index, name in model_names.items() if name == "apple"}
    green_ids = {index for index, name in model_names.items() if name == "green_apple"}
    yellow_ids = {index for index, name in model_names.items() if name == "yellow_apple"}
    if not apple_ids and not (green_ids or yellow_ids):
        raise RuntimeError(f"The automatic-label model has no apple class: {model_names}")

    rows, totals = [], {"green_apple": 0, "yellow_apple": 0, "empty_images": 0}
    for image_path in tqdm(raw_frames, desc="Automatic apple labels"):
        label_path = paths.auto_labels / f"{image_path.stem}.txt"
        preview_path = paths.auto_previews / image_path.name
        if label_path.exists() and preview_path.exists() and not overwrite:
            boxes = fp.read_yolo_labels(label_path)
            rows.append({"image": image_path.name, "status": "reused", "boxes": len(boxes)})
            continue

        image = cv2.imread(str(image_path))
        if image is None:
            raise OSError(f"Cannot read {image_path}")
        result = model.predict(image, conf=AUTO_CONFIDENCE, imgsz=AUTO_IMAGE_SIZE, device=AUTO_DEVICE, verbose=False)[0]
        boxes, details = [], []
        if result.boxes is not None:
            for candidate in result.boxes:
                source_id = int(candidate.cls[0].item())
                if source_id not in apple_ids | green_ids | yellow_ids:
                    continue
                xyxy = tuple(float(value) for value in candidate.xyxy[0].tolist())
                if source_id in green_ids:
                    class_id, colour = fp.GREEN_APPLE, {"source": "model_green_apple"}
                elif source_id in yellow_ids:
                    class_id, colour = fp.YELLOW_APPLE, {"source": "model_yellow_apple"}
                else:
                    class_id, colour = classify_box_colour(image, xyxy)
                    colour["source"] = "COCO_apple_plus_HSV"
                boxes.append(fp.xyxy_to_yolo(class_id, xyxy, image.shape[1], image.shape[0]))
                totals[CLASS_NAMES[class_id]] += 1
                details.append({
                    "class_name": CLASS_NAMES[class_id],
                    "detector_confidence": round(float(candidate.conf[0].item()), 5),
                    **colour,
                })
        fp.write_yolo_labels(label_path, boxes)  # empty files preserve automatic negative examples.
        cv2.imwrite(str(preview_path), fp.draw_yolo_boxes(image, boxes))
        if not boxes:
            totals["empty_images"] += 1
        rows.append({"image": image_path.name, "status": "generated", "boxes": len(boxes), "detections": details})

    manifest_path = paths.manifests / "automatic_labels.jsonl"
    manifest_path.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")
    summary = {"model": str(model_source), "images": len(raw_frames), **totals, "labels": str(paths.auto_labels)}
    (paths.manifests / "automatic_label_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary

RUN_AUTOMATIC_LABELLING = False
if RUN_AUTOMATIC_LABELLING:
    auto_summary = automatic_apple_labels(AUTO_MODEL, overwrite=REBUILD_AUTO_LABELS)
    print(json.dumps(auto_summary, indent=2))
else:
    print("[autolabel] Set RUN_AUTOMATIC_LABELLING=True to generate labels with", AUTO_MODEL)


In [ ]:
# %% 4) Automatic-label counts and preview locations (no manual label step)
auto_label_files = sorted(paths.auto_labels.glob("*.txt"))
auto_preview_files = sorted(paths.auto_previews.glob("*.jpg"))
print(f"[auto labels] files={len(auto_label_files)}/{len(raw_frames)} | previews={len(auto_preview_files)}")
if len(auto_label_files) == len(raw_frames):
    class_counts = {name: 0 for name in CLASS_NAMES}
    for label_file in auto_label_files:
        for box in fp.read_yolo_labels(label_file):
            class_counts[CLASS_NAMES[box.class_id]] += 1
    print("[auto labels] boxes:", class_counts)
else:
    print("[auto labels] Run cell 3 before splitting the dataset.")


In [ ]:
# %% 5) Split ORIGINAL automatically labelled frames into train/val/test = 70/10/20
# The temporal split comes BEFORE augmentation to prevent leakage of video-neighbour/augmented copies.
REBUILD_YOLO_SPLIT = False

if REBUILD_YOLO_SPLIT:
    for split in ("train", "val", "test"):
        for folder, pattern in ((paths.yolo_images / split, "*.jpg"), (paths.yolo_labels / split, "*.txt")):
            for generated_file in folder.glob(pattern):
                generated_file.unlink()
if len(list(paths.auto_labels.glob("*.txt"))) != len(raw_frames):
    raise RuntimeError("Automatic labels are incomplete. Run cell 3 with RUN_AUTOMATIC_LABELLING=True first.")

split_summary = fp.build_temporal_yolo_dataset(
    paths,
    train_fraction=0.70,
    val_fraction=0.10,
    label_directory=paths.auto_labels,
)
print("[split]", split_summary)
print("[data yaml]", paths.yolo_yaml)


In [ ]:
# %% 6) Augment TRAINING frames only; transformed automatic labels are written with each image.
# Includes horizontal view reversal, low/high light, motion blur, and synthetic leaf occlusion.
# Hue shifting is excluded because hue defines green_apple versus yellow_apple.
REBUILD_AUGMENTATIONS = False
augmentation_summary = fp.augment_training_split(paths, seed=42, overwrite=REBUILD_AUGMENTATIONS)
print("[augment]", augmentation_summary)


In [ ]:
# %% 7) Dataset audit: image/label parity and automatic green/yellow class balance
dataset_report = fp.validate_yolo_dataset(paths)
print(json.dumps(dataset_report, indent=2))
assert dataset_report["valid"], "Fix the dataset audit before training."


In [ ]:
# %% 8) AAM — Attribute Attention Module (colour + shape + scale fusion)
import torch
import torch.nn as nn
import torch.nn.functional as F

class AAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.colour = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.SiLU(), nn.Conv2d(hidden, channels, 1)
        )
        self.shape = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.scale = nn.Sequential(nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(channels, channels, 1))
        self.weights = nn.Parameter(torch.zeros(3))

    def forward(self, features):
        colour = self.colour(features).expand_as(features)
        shape = self.shape(features)
        scale = self.scale(features)
        alpha = torch.softmax(self.weights, dim=0)
        attention = torch.sigmoid(alpha[0] * colour + alpha[1] * shape + alpha[2] * scale)
        return features * attention


In [ ]:
# %% 9) EBCA — efficient bidirectional P3 <-> P4 cross-attention for overlapping apples
class EBCA(nn.Module):
    def __init__(self, p3_channels, p4_channels):
        super().__init__()
        self.p4_to_p3 = nn.Conv2d(p4_channels, p3_channels, 1, bias=False)
        self.p3_to_p4 = nn.Conv2d(p3_channels, p4_channels, 1, bias=False)
        self.p3_query = nn.Conv2d(p3_channels, p3_channels, 1, bias=False)
        self.p4_query = nn.Conv2d(p4_channels, p4_channels, 1, bias=False)
        self.p3_norm = nn.BatchNorm2d(p3_channels)
        self.p4_norm = nn.BatchNorm2d(p4_channels)

    def forward(self, p3, p4):
        p4_up = F.interpolate(self.p4_to_p3(p4), size=p3.shape[-2:], mode="bilinear", align_corners=False)
        p3_gate = torch.sigmoid(self.p3_query(p3) * p4_up / math.sqrt(p3.shape[1]))
        enhanced_p3 = self.p3_norm(p3 + p3_gate * p4_up)
        p3_down = F.adaptive_avg_pool2d(self.p3_to_p4(p3), p4.shape[-2:])
        p4_gate = torch.sigmoid(self.p4_query(p4) * p3_down / math.sqrt(p4.shape[1]))
        enhanced_p4 = self.p4_norm(p4 + p4_gate * p3_down)
        return enhanced_p3, enhanced_p4


In [ ]:
# %% 10) FM — focal modulation for partial occlusion and contextual background suppression
class FM(nn.Module):
    def __init__(self, channels, levels=3):
        super().__init__()
        self.query = nn.Conv2d(channels, channels, 1)
        self.context = nn.ModuleList(
            [nn.Conv2d(channels, channels, 3 + 2 * level, padding=1 + level, groups=channels) for level in range(levels)]
        )
        self.gates = nn.Conv2d(channels, levels + 1, 1)
        self.modulator = nn.Conv2d(channels, channels, 1)
        self.output = nn.Conv2d(channels, channels, 1)

    def forward(self, features):
        query = self.query(features)
        gates = torch.softmax(self.gates(features), dim=1)
        context, current = 0, features
        for level, layer in enumerate(self.context):
            current = layer(current)
            context = context + gates[:, level:level + 1] * current
        context = context + gates[:, -1:] * F.adaptive_avg_pool2d(features, 1)
        return self.output(query * self.modulator(context))


In [ ]:
# %% 11) KWConv — adaptive 3x3/5x5/7x7 kernel warehouse for small and large apples
class KWConv(nn.Module):
    def __init__(self, channels, kernels=(3, 5, 7)):
        super().__init__()
        self.branches = nn.ModuleList(
            [nn.Conv2d(channels, channels, kernel, padding=kernel // 2, bias=False) for kernel in kernels]
        )
        self.selector = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, len(kernels), 1))
        self.norm = nn.BatchNorm2d(channels)
        self.activation = nn.SiLU()

    def forward(self, features):
        weights = torch.softmax(self.selector(features), dim=1)
        mixed = sum(weights[:, i:i + 1] * branch(features) for i, branch in enumerate(self.branches))
        return self.activation(self.norm(mixed))


In [ ]:
# %% 12) Proposed enhancement neck and interface test: AAM -> EBCA -> FM -> KWConv(P3,P4,P5)
class ProposedEnhancementNeck(nn.Module):
    def __init__(self, p3_channels, p4_channels, p5_channels):
        super().__init__()
        self.aam = AAM(p4_channels)
        self.ebca = EBCA(p3_channels, p4_channels)
        self.fm = FM(p4_channels)
        self.kw_p3, self.kw_p4, self.kw_p5 = KWConv(p3_channels), KWConv(p4_channels), KWConv(p5_channels)

    def forward(self, p3, p4, p5):
        p4 = self.aam(p4)
        p3, p4 = self.ebca(p3, p4)
        p4 = self.fm(p4)
        return self.kw_p3(p3), self.kw_p4(p4), self.kw_p5(p5)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
proposed_neck = ProposedEnhancementNeck(96, 192, 384).to(DEVICE).eval()
with torch.inference_mode():
    neck_outputs = proposed_neck(
        torch.zeros(1, 96, 80, 80, device=DEVICE),
        torch.zeros(1, 192, 40, 40, device=DEVICE),
        torch.zeros(1, 384, 20, 20, device=DEVICE),
    )
architecture_report = {"device": DEVICE, "output_shapes": [list(x.shape) for x in neck_outputs]}
(paths.results / "architecture_smoke_test.json").write_text(json.dumps(architecture_report, indent=2), encoding="utf-8")
print("[architecture]", architecture_report)


In [ ]:
# %% 13) Train the proposed AAM + EBCA + FM + KWConv YOLOv8 model on automatic labels
# A custom Ultralytics trainer inserts the proposed P3/P4/P5 enhancement after YOLO reconstructs its two-class model.
# This guarantees the saved checkpoint contains the AAM–EBCA–FM–KWConv neck.

RUN_PROPOSED_TRAINING = False
RUN_BASELINE_TRAINING = False
TRAIN_EPOCHS, TRAIN_IMAGE_SIZE = 100, 1024
TRAIN_DEVICE = 0 if torch.cuda.is_available() else "cpu"

if RUN_PROPOSED_TRAINING:
    training_result = fp.train_proposed_yolov8(
        paths, model_name="yolov8n.pt", epochs=TRAIN_EPOCHS, image_size=TRAIN_IMAGE_SIZE,
        batch=-1, device=TRAIN_DEVICE,
    )
    print(training_result)
elif RUN_BASELINE_TRAINING:
    training_result = fp.train_baseline_yolov8(
        paths, model_name="yolov8n.pt", epochs=TRAIN_EPOCHS, image_size=TRAIN_IMAGE_SIZE,
        batch=-1, device=TRAIN_DEVICE,
    )
    print(training_result)
else:
    print("[train] Set RUN_PROPOSED_TRAINING=True for the proposed model, or RUN_BASELINE_TRAINING=True for ablation.")


In [ ]:
# %% 14) Validate model accuracy: mAP50-95, precision, recall, and confusion matrix -> Results/validation/
PROPOSED_WEIGHTS = paths.training_results / "aam_ebca_fm_kwconv_auto" / "weights" / "best.pt"
BASELINE_WEIGHTS = paths.training_results / "yolov8_green_yellow_apple" / "weights" / "best.pt"
BEST_WEIGHTS = PROPOSED_WEIGHTS if PROPOSED_WEIGHTS.exists() else BASELINE_WEIGHTS
RUN_VALIDATION = False
if RUN_VALIDATION:
    assert BEST_WEIGHTS.exists(), f"Missing trained weights: {BEST_WEIGHTS}"
    validation_result = fp.validate_detector(paths, BEST_WEIGHTS, image_size=1024)
    print(validation_result)
else:
    print("[validation] Set RUN_VALIDATION=True after baseline/proposed detector training.")


In [ ]:
# %% 15) AdaBins pixel-wise depth (relative orchard depth; no metric-depth claim without calibration)
USE_ADABINS = False
ADABINS_REPOSITORY = ROOT / "third_party" / "AdaBins"
depth_estimator = fp.AdaBinsDepth(ADABINS_REPOSITORY, dataset="kitti") if USE_ADABINS else None
print("[depth]", "AdaBins ready" if depth_estimator is not None else "disabled; set USE_ADABINS=True when ready")


In [ ]:
# %% 16) Optional depth preview for one automatic-label frame -> Results/depth_preview.png
DEPTH_PREVIEW_FRAME = None  # Example: raw_frames[0]
if depth_estimator is not None and DEPTH_PREVIEW_FRAME is not None:
    frame = cv2.imread(str(DEPTH_PREVIEW_FRAME))
    depth_map = depth_estimator.predict(frame)
    normalised = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    colour_depth = cv2.applyColorMap(normalised, cv2.COLORMAP_INFERNO)
    panel = np.hstack((frame, colour_depth))
    output = paths.results / "depth_preview.png"
    cv2.imwrite(str(output), panel)
    print("[depth preview]", output)
else:
    print("[depth preview] Set USE_ADABINS=True and DEPTH_PREVIEW_FRAME=raw_frames[0].")


In [ ]:
# %% 17) Depth-aware ByteTrack-style tracking, once-only line counting, and Results outputs
# High-confidence matching occurs first; low-confidence detections recover temporarily missed fruit.
RUN_FINAL_TRACKING = False
if RUN_FINAL_TRACKING:
    assert BEST_WEIGHTS.exists(), f"Missing trained weights: {BEST_WEIGHTS}"
    counting_line = (
        (preflight["width"] // 2, 0),
        (preflight["width"] // 2, preflight["height"] - 1),
    )
    tracking_summary = fp.run_depth_aware_tracking(
        paths,
        weights=BEST_WEIGHTS,
        adabins=depth_estimator,
        low_confidence=0.10,
        image_size=1024,
        counting_line=counting_line,
        count_direction="both",
    )
    print(json.dumps(tracking_summary, indent=2))
else:
    print("[tracking] Set RUN_FINAL_TRACKING=True after training. AdaBins is optional.")


In [ ]:
# %% 18) Final artifact list — all generated data stays in Dataset/ and results stay in Results/
for folder in (paths.auto_labels, paths.augmented_images, paths.results):
    files = sorted(path.relative_to(ROOT) for path in folder.rglob("*") if path.is_file())
    print(f"\n[{folder.relative_to(ROOT)}] {len(files)} files")
    for path in files[:20]:
        print(" -", path)
    if len(files) > 20:
        print(" - ...")
